# stareControl — リアルタイム視線推定・9点学習

ブラウザのWebカメラ映像を**リアルタイム**で解析する実験版です。録画は不要です。計算はブラウザ内のMediaPipe Face Landmarker（Web API）で行い、ColabのPythonランタイムと毎フレーム通信しません。

## 実行方法
1. 下のセルを実行します（pipインストール不要）。
2. 表示された「**カメラを開始**」を押し、カメラを許可します。
3. 「**全画面**」→「**9点学習**」を押し、順番に表示される点を約2秒ずつ見つめます。顔の位置はなるべく固定してください。
4. 学習が終わると、推定した視線位置を**緑色の点**で表示します。
5. **両目を軽く細めた状態を約0.45秒維持**すると、疑似クリックします。短い通常の瞬きは無視します。感度・維持時間をスライダーで変更できます。左/右ボタンか `R` キーで疑似クリック種別を切り替えます。
6. 必要に応じてCSVを保存し、「終了」を押してカメラを停止します。

**注意**: 視線位置は**9点から推定した近似座標**です。通常のWebカメラでは視線の精度に限界があり、顔の移動や照明条件でずれます。細目判定の閾値は個人差があります。今回のクリックは画面内の**疑似クリック**であり、OSのマウス操作ではありません。

**プライバシー**: カメラ映像を録画・保存・アップロードするコードはありません。外部CDNからMediaPipeのライブラリとモデルを読み込みます。CSV保存は明示的に押したときのみ行います。

In [ ]:
# Colabのブラウザ上でリアルタイムの視線推定UIを起動します。
# 初回はMediaPipeのJavaScript/WASMとモデルをネットワークから読み込みます。
import urllib.request
from IPython.display import display, Javascript

URL = 'https://raw.githubusercontent.com/ryouy/stareControl/main/web/realtime.js'
req = urllib.request.Request(URL, headers={'Cache-Control': 'no-cache'})
with urllib.request.urlopen(req, timeout=30) as r:
    script = r.read().decode('utf-8')
display(Javascript(script))
print('画面上の「カメラを開始」を押してください。')

### 補足
- 再度セルを実行すると既存のカメラを停止して新しい画面を開きます。
- 学習はその都度やり直せます。学習済みモデルはメモリ内のみで、ページ更新後は再学習が必要です。
- 「全画面」にする前と後で画面の座標系が変わるため、**全画面にした後に9点学習**してください。
- Colabの実行環境によってはブラウザ側のカメラ/全画面表示が制限される場合があります。カメラ権限とブラウザの設定を確認してください。
- 今回はモデルの認識精度を調べるための実験であり、物理的なカーソル制御は次のデスクトップ版で実装します。